<a href="https://colab.research.google.com/github/darianneer/TravelAgentChatbot/blob/main/TravelAgentChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.3 MB/s eta 0:00:00


In [9]:
import os
from typing import List, Dict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.schema import HumanMessage, SystemMessage, AIMessage
from google.colab import userdata

In [18]:
os.environ["GOOGLE_API_KEY"]=userdata.get('GEMINI_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, streaming=True) # Enables token by token

summarizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2, streaming=False)

In [19]:
system_prompt="""You are a professional travel agent.
You ONLY answer travel-related questions: flights, hotels, destinations, trips, itineraries, visas, packing and travel tips.
If the user asks anything non-travel-related, you MUST respond with exactly:
'I can't help with it.'
Do not explain this rule to the user.
"""

In [20]:
travel_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=system_prompt),
        HumanMessage(content="{history}\n\nUser: {user_input}"),
    ]
)

In [21]:
summary_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are a helpful assisstant that summarizes chat histories."
         ),
        ("human",
         "Summarize the following travel-related conversation briefly but clearly,"
         "preserving important user preferences and decisions:\n\n{history}")
    ]
)

In [22]:
chat_history: List = []

user_turn_count=0

def history_to_text(history: List) -> str:
  lines = []
  for msg in history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    if isinstance(msg, AIMessage):
      role = "System"
    lines.append(f"{role}: {msg.content}")
  return "\n".join(lines)

def summarize_history(history: List) -> List:
  if not history:
    return history

  history_text = history_to_text(history)
  summary_prompt = summary_prompt_template.format_prompt(history=history_text)
  summary_response = summarizer_llm.invoke(summary_prompt.to_messages())

  summary_message = SystemMessage(
      content=f"Conversation summary:\n{summary_response.content}"
  )

  return [summary_message]

In [23]:
def chat():
  global chat_history, user_turn_count

  print("Travel Agent Chatbot(type 'exit' to quit)\n")

  while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
      print("Chatbot: Goodbye and safe travels!")
      break

    user_msg = HumanMessage(content=user_input)
    chat_history.append(user_msg)
    user_turn_count += 1

    history_text = history_to_text(chat_history[:-1])

    prompt = travel_prompt_template.format_prompt(
        history=history_text, user_input=user_input,)

    print("Chatbot: ", end="")
    full_response = []
    for chunk in llm.stream(prompt.to_messages()):
      token = chunk.content
      print(token, end="", flush=True)
      full_response.append(token)
    print()

    bot_text = "".join(full_response)
    bot_msg = AIMessage(content=bot_text)
    chat_history.append(bot_msg)

    if user_turn_count % 5 == 0:
      chat_history = summarize_history(chat_history)

if __name__ == "__main__":
  chat()

Travel Agent Chatbot(type 'exit' to quit)

You: hi
Chatbot: I can help you with your travel plan! To get started, could you please tell me:

*   **Where are you planning to go?** (Destination)
*   **When are you planning to travel?** (Dates or time of year)
*   **How long will your trip be?**
*   **What kind of trip are you looking for?** (e.g., relaxing beach, adventurous hiking, cultural city break, family vacation, solo travel)
*   **What's your estimated budget?** (e.g., budget-friendly, mid-range, luxury)
*   **Who are you traveling with?** (e.g., solo, couple, family with kids, friends)

The more details you provide, the better I can tailor a plan for you!
You: I want to learn AI
Chatbot: I can't help with it.
You: What can you help with?
Chatbot: 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 45.175472708s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [24]:
import unittest.mock
import io
import sys

def run_test_chat(queries):
    global chat_history, user_turn_count

    # Reset chat history and turn count for the test
    chat_history = []
    user_turn_count = 0

    # Add 'exit' to the end of queries to terminate the chat loop gracefully
    test_queries = list(queries) + ['exit']
    query_iterator = iter(test_queries)

    def mock_input(prompt):
        try:
            query = next(query_iterator)
            print(f"You: {query}") # Print the 'user input' during the mock
            return query
        except StopIteration:
            return 'exit' # Should not happen if 'exit' is added properly

    captured_output = io.StringIO()
    with unittest.mock.patch('builtins.input', mock_input):
        with sys.stdout as original_stdout:
            sys.stdout = captured_output
            chat()
            sys.stdout = original_stdout # Restore stdout immediately

    return captured_output.getvalue(), chat_history

# Define your test queries
test_queries = [
    "Hi",
    "Travel to Paris",
    "What is the capital of France?", # Non-travel
    "I need a hotel in London for 3 nights next month",
    "Tell me about the weather tomorrow", # Non-travel
    "What should I pack for a beach vacation?",
    "flight from new york to tokyo"
]

print("\n--- Running Chatbot Test Suite ---")
test_output, final_history = run_test_chat(test_queries)
print(test_output)
print("--- Final Chat History ---")
for msg in final_history:
    print(msg)



--- Running Chatbot Test Suite ---


AttributeError: 'OutStream' object has no attribute 'watch_fd_thread'